In [ ]:
import os
import pyomo.environ as pe
from egret.parsers.matpower_parser import create_ModelData

# Import the solver function and the specific Rectangular (QCQP) model generator
from egret.models.acopf import solve_acopf, create_rsv_acopf_model



In [ ]:
# -----------------------------------------------------------------------------
# 1. Load your MATPOWER Data
# -----------------------------------------------------------------------------
# Point this to the .m file you used to generate your data earlier
matpower_file = '../pglib-opf-21.07/pglib_opf_case14_ieee.m' 

print(f"Loading data from {matpower_file}...")
model_data = create_ModelData(matpower_file)

# -----------------------------------------------------------------------------
# 2. Build and Solve the QCQP Model
# -----------------------------------------------------------------------------
# We use 'create_rsv_acopf_model' (Rectangular Standard Variables).
# This strips out all sin() and cos() functions, replacing them with Vr and Vj,
# natively forming a strict Quadratically Constrained Quadratic Program (QCQP).

print("Building and solving Rectangular QCQP ACOPF with IPOPT...")
md_solved, pyomo_model, results = solve_acopf(
    model_data,
    solver="ipopt",
    acopf_model_generator=create_rsv_acopf_model, 
    return_model=True,
    return_results=True,
    solver_tee=True, # Set to True to see the IPOPT iteration log
    include_feasibility_slack=False
)

# -----------------------------------------------------------------------------
# 3. Extract and Verify Results
# -----------------------------------------------------------------------------
from pyomo.opt import TerminationCondition

if results.solver.termination_condition == TerminationCondition.optimal:
    print("\n--- Optimal Solution Found ---")
    
    # Egret automatically maps the solved Pyomo variables back into a clean dictionary
    total_cost = md_solved.data['system']['total_cost']
    print(f"Total Generation Cost: {total_cost:.4f}")
    
    # Extract Bus Voltages
    print("\n--- Voltage Results (First 5 Buses) ---")
    buses = dict(md_solved.elements(element_type='bus'))
    
    for i, (bus_id, bus_dict) in enumerate(buses.items()):
        if i >= 5: 
            break
            
        # Egret converts the rectangular variables (Vr, Vj) back to 
        # polar magnitudes (Vm) and angles (Va) in degrees for you automatically.
        vm = bus_dict['vm']
        va = bus_dict['va']
        print(f"Bus {bus_id}: Vm = {vm:.4f} p.u. | Va = {va:.4f} deg")
        
    # Extract Generator Setpoints
    print("\n--- Generation Dispatch (First 5 Generators) ---")
    gens = dict(md_solved.elements(element_type='generator'))
    
    for i, (gen_id, gen_dict) in enumerate(gens.items()):
        if i >= 5: 
            break
        pg = gen_dict['pg']
        qg = gen_dict['qg']
        print(f"Gen {gen_id}: Pg = {pg:.4f} p.u. | Qg = {qg:.4f} p.u.")

else:
    print("\nSolver failed to converge.")
    print("Status:", results.solver.status)

Loading data from ../pglib-opf-21.07/pglib_opf_case14_ieee.m...
Building and solving Rectangular QCQP ACOPF with IPOPT...
Ipopt 3.11.1: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

NOTE: You are using Ipopt by default with the MUMPS linear solver.
      Other linear solvers might be more efficient (see Ipopt documentation).


This is Ipopt version 3.11.1, running with linear solver mumps.

Number of nonzeros in equality constraint Jacobian...:      611
Number of nonzeros in inequality constraint Jacobian.:      160
Number of nonzeros in Lagrangian Hessian.............:      183

Total number of variables............................:      168
      